# 1. Import libraries & Config params

## 1.1 Import libraries

In [1]:
!pip install torchstain

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from torchvision import models, transforms
import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    balanced_accuracy_score, confusion_matrix, classification_report
)

from typing import Literal
from tqdm.auto import tqdm
import random
import os, gc
from datetime import datetime
from time import time
from collections import Counter
from torchstain.torch.normalizers.macenko import TorchMacenkoNormalizer
import warnings

warnings.filterwarnings('ignore')

# Verify environment
print(f'Pytorch Version: {torch.__version__}')
print(f'Cuda is available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPUs available: {torch.cuda.device_count()}')

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Pytorch Version: 2.8.0+cu126
Cuda is available: True
GPU: Tesla T4
GPUs availabel: 2


## 1.2 Config params

In [ ]:
# Cell 2: Configuration and Directory Paths

# Directory paths
DIR_PHASE_1 = 'wbc-isbi/phase1'
DIR_PHASE_2_TRAIN = 'wbc-isbi/phase2/train'
DIR_PHASE_2_EVAL = 'wbc-isbi/phase2/eval'
DIR_PHASE_2_TEST = 'wbc-isbi/phase2/test'

LABEL_PATH_PHASE_1 = 'wbc-isbi/phase1_label.csv'
LABEL_PATH_PHASE_2_TRAIN = 'wbc-isbi/phase2_train.csv'
LABEL_PATH_PHASE_2_EVAL = 'wbc-isbi/phase2_eval.csv'
LABEL_PATH_PHASE_2_TEST = 'wbc-isbi/phase2_test.csv'

# Class information
NUM_CLASSES = 13
CLASS_NAMES = ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

# Configuration 1: ResNet50 with patience=10
config1 = {
    'model': {
        'name': 'resnet50',
        'pretrained': True,
        'num_classes': NUM_CLASSES,
        'head': {
            'type': 'custom',  # default / custom
            'hidden_dim': 512,
            'dropout': 0.0,
            'note': "Add macenko normalize",
        },
        'normalize': {
            'mean': (0.485, 0.456, 0.406),
            'std': (0.229, 0.224, 0.225),
        }
    },
    'data': {
        'use_chula': False,
        'image_size': 368,
        'batch_size': 256,
        'num_workers': 2,
        'val_split': 0.2,
        'seed': 42,
        'boost_rare_classes': {
            'enable': False,
            'rare_classes': None,
            'boost_factor': 1,
        },
    },
    'training': {
        'stages': [
            {
                'name': 'backbone',
                'freeze_backbone': False,
                'epochs': 100,
                'loss': {
                    'name': 'cross_entropy',  # cross_entropy / balanced_softmax
                    'label_smoothing': 0.0,
                },
                'sampler': {
                    'name': 'random_sampler',  # random_sampler / class_balanced_sampler / square_root_sampler
                    'replacement': False,
                },
                'optimizer': {
                    'name': 'AdamW',  # SGD / AdamW
                    'lr': 1e-4,
                    'weight_decay': 0.0,
                },
                'early_stopping': {
                    'enable': True,
                    'patience': 10,
                },
            },
            {
                'name': 'classifier',
                'freeze_backbone': True,
                'epochs': 50,
                'loss': {
                    'name': 'combined_loss',  # cross_entropy / weighted_cross_entropy / balanced_softmax / combined_loss
                    'label_smoothing': 0.0,
                },
                'sampler': {
                    'name': 'class_balanced_sampler',  # random_sampler / class_balanced_sampler / square_root_sampler
                    'replacement': True,
                },
                'optimizer': {
                    'name': 'AdamW',  # SGD / AdamW
                    'lr': 1e-3,
                    'weight_decay': 0.0,
                },
                'early_stopping': {
                    'enable': True,
                    'patience': 10,
                },
            },
        ]
    }
}

# Configuration 2: ResNet152 with patience=5 for backbone stage
config2 = {
    'model': {
        'name': 'resnet152',
        'pretrained': True,
        'num_classes': NUM_CLASSES,
        'head': {
            'type': 'custom',
            'hidden_dim': 512,
            'dropout': 0.0,
            'note': "Add macenko normalize",
        },
        'normalize': {
            'mean': (0.485, 0.456, 0.406),
            'std': (0.229, 0.224, 0.225),
        }
    },
    'data': {
        'use_chula': False,
        'image_size': 368,
        'batch_size': 256,
        'num_workers': 2,
        'val_split': 0.2,
        'seed': 42,
        'boost_rare_classes': {
            'enable': False,
            'rare_classes': None,
            'boost_factor': 1,
        },
    },
    'training': {
        'stages': [
            {
                'name': 'backbone',
                'freeze_backbone': False,
                'epochs': 100,
                'loss': {
                    'name': 'cross_entropy',
                    'label_smoothing': 0.0,
                },
                'sampler': {
                    'name': 'random_sampler',
                    'replacement': False,
                },
                'optimizer': {
                    'name': 'AdamW',
                    'lr': 1e-4,
                    'weight_decay': 0.0,
                },
                'early_stopping': {
                    'enable': True,
                    'patience': 5,  # Different from config1
                },
            },
            {
                'name': 'classifier',
                'freeze_backbone': True,
                'epochs': 50,
                'loss': {
                    'name': 'combined_loss',
                    'label_smoothing': 0.0,
                },
                'sampler': {
                    'name': 'class_balanced_sampler',
                    'replacement': True,
                },
                'optimizer': {
                    'name': 'AdamW',
                    'lr': 1e-3,
                    'weight_decay': 0.0,
                },
                'early_stopping': {
                    'enable': True,
                    'patience': 10,
                },
            },
        ]
    }
}

# List of configurations for training
CONFIGS = [
    ('resnet50', config1),
    ('resnet152', config2),
]

print(f"Training {len(CONFIGS)} models:")
for name, cfg in CONFIGS:
    num_stages = len(cfg['training']['stages'])
    print(f"  - {name}: {num_stages} stages")

SAVE_DIR = './checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device is {device}!")
print(f"Configuration completed!")

Training with 2 stages
Device is cuda!
Setting completed!


# 2. EDA

# 3. Data processing

## 3.1 Offline process images with macenko normalization

In [ ]:
def pil_to_tensor_uint8(img):
    """Convert PIL image to uint8 tensor [3, H, W]."""
    x = torch.from_numpy(np.array(img))
    return x.permute(2, 0, 1)  # [3, H, W], uint8

# Initialize normalizer (CPU)
normalizer = TorchMacenkoNormalizer()

ref_paths = ["wbc-isbi/phase1/00223657.jpg"]

for p in ref_paths:
    img = Image.open(p).convert("RGB")
    x = pil_to_tensor_uint8(img)
    normalizer.fit(x)

def macenko_lambda(img: Image.Image):
    """Apply Macenko stain normalization to a PIL image."""
    # PIL → torch uint8 [3, H, W]
    x = pil_to_tensor_uint8(img)

    try:
        # --- Quick stain validity check ---
        # Optical Density
        OD = -torch.log((x.float() + 1) / 255)

        # Number of pixels with stain (threshold same as torchstain)
        valid_pixels = (OD.sum(dim=0) > 0.15).sum().item()

        # If too few stained pixels → skip Macenko
        if valid_pixels < 50:
            return img  # Fallback: original image

        # --- Macenko normalize ---
        Inorm, _, _ = normalizer.normalize(x, stains=False)

        # Clamp & convert to PIL
        Inorm = Inorm.clamp(0, 255).byte()      # [3, H, W]
        Inorm = Inorm.permute(1, 2, 0)          # [H, W, 3]
        Inorm_np = Inorm.cpu().numpy()

        return Image.fromarray(Inorm_np)

    except Exception as e:
        # Absolute safety fallback
        return img

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Source directories (original images) - INCLUDING TEST SET
SRC_DIRS = [
    DIR_PHASE_1,
    DIR_PHASE_2_TRAIN,
    DIR_PHASE_2_EVAL,
    DIR_PHASE_2_TEST,
]

# Destination directory (normalized images)
DST_BASE_DIR = "MACENKO_NORMALIZE"

# Number of workers
NUM_WORKERS = 4


def process_single_image(args):
    """Process a single image."""
    src_path, dst_path = args
    try:
        # Skip if already processed
        if os.path.exists(dst_path):
            return True, src_path, "skipped"
        
        img = Image.open(src_path).convert('RGB')
        img_normalized = macenko_lambda(img)
        
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        img_normalized.save(dst_path, quality=95)
        
        return True, src_path, "processed"
    except Exception as e:
        return False, src_path, str(e)


def collect_image_tasks(src_dirs, dst_base_dir):
    """Collect all images from directories."""
    tasks = []
    
    print("📁 Collecting images from directories...")
    for src_dir in src_dirs:
        if not os.path.exists(src_dir):
            print(f"⚠️ Does not exist: {src_dir}")
            continue
            
        dir_name = os.path.basename(src_dir)
        files = [f for f in os.listdir(src_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))]
        print(f"   ✅ {dir_name}: {len(files)} images")
        
        for img_name in files:
            src_path = os.path.join(src_dir, img_name)
            dst_path = os.path.join(dst_base_dir, dir_name, img_name)
            tasks.append((src_path, dst_path))
    
    return tasks


def run_macenko_offline(src_dirs, dst_base_dir, num_workers=4):
    """Run Macenko offline processing."""
    
    tasks = collect_image_tasks(src_dirs, dst_base_dir)
    print(f"\n📊 Total images to process: {len(tasks)}")
    
    if len(tasks) == 0:
        print("❌ No images found!")
        return
    
    os.makedirs(dst_base_dir, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    failed_files = []
    
    print(f"🔧 Using ThreadPool with {num_workers} workers\n")
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_image, task): task for task in tasks}
        
        pbar = tqdm(as_completed(futures), total=len(tasks), desc="Macenko Processing")
        for future in pbar:
            success, path, status = future.result()
            
            if success:
                if status == "skipped":
                    skip_count += 1
                else:
                    success_count += 1
            else:
                fail_count += 1
                failed_files.append((path, status))
            
            pbar.set_postfix({'done': success_count, 'skip': skip_count, 'fail': fail_count})
    
    # Report
    print(f"\n{'='*50}")
    print(f"📊 MACENKO OFFLINE PROCESSING RESULTS")
    print(f"{'='*50}")
    print(f"✅ Successfully processed: {success_count}")
    print(f"⏭️  Skipped: {skip_count}")
    print(f"❌ Errors: {fail_count}")
    print(f"📁 Output: {dst_base_dir}")
    
    if failed_files:
        print(f"\n⚠️ Failed files (showing max 10):")
        for path, error in failed_files[:10]:
            print(f"   {os.path.basename(path)}: {error}")
        if len(failed_files) > 10:
            print(f"   ... and {len(failed_files) - 10} other files")
    
    return dst_base_dir


# Run processing
print("🚀 Starting Macenko Offline Processing (including Test Set)...")
run_macenko_offline(
    src_dirs=SRC_DIRS,
    dst_base_dir=DST_BASE_DIR,
    num_workers=NUM_WORKERS
)

# Verify results
print("📁 Checking output directories:\n")

for src_dir in SRC_DIRS:
    dir_name = os.path.basename(src_dir)
    dst_dir = os.path.join(DST_BASE_DIR, dir_name)
    
    if os.path.exists(dst_dir):
        dst_count = len(os.listdir(dst_dir))
        src_count = len(os.listdir(src_dir)) if os.path.exists(src_dir) else 0
        status = "✅" if dst_count == src_count else "⚠️"
        print(f"{status} {dir_name}: {dst_count}/{src_count} images")
    else:
        print(f"❌ {dir_name}: Not found")

## 3.2 Define transforms, dataset, dataloader

In [ ]:
# Update directory paths to use normalized images
DIR_PHASE_1 = 'MACENKO_NORMALIZE/phase1'
DIR_PHASE_2_TRAIN = 'MACENKO_NORMALIZE/train'
DIR_PHASE_2_EVAL = 'MACENKO_NORMALIZE/eval'
DIR_PHASE_2_TEST = 'MACENKO_NORMALIZE/test'

def set_seed(seed=42):
    """Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    """Worker seed function for DataLoader."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Set seed using config1 (same seed for both)
SEED = config1['data']['seed']
set_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

def create_transforms(config, split: Literal["train", "val"]):
    """Create image transforms for training or validation."""
    if split == 'train':
        train_transforms = transforms.Compose([
            transforms.Resize((config['data']['image_size'], config['data']['image_size'])),
            transforms.CenterCrop(224),
            transforms.RandAugment(),
            transforms.ToTensor(),
            transforms.Normalize(mean=config['model']['normalize']['mean'],
                                 std=config['model']['normalize']['std']),
        ])
        return train_transforms
    elif split == 'val':
        val_transforms = transforms.Compose([
            transforms.Resize((config['data']['image_size'], config['data']['image_size'])),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=config['model']['normalize']['mean'],
                                 std=config['model']['normalize']['std']),
        ])
        return val_transforms
    else:
        raise ValueError(f"Split must be either 'train' or 'val'")


class WBCDataset(Dataset):
    """
    WBC Dataset with dynamic class mapping.

    Args:
        df: DataFrame with 'ID' and 'labels' columns
        img_dirs: Directory or list of directories containing images
        transform: Image transforms
        class_to_idx: Dictionary mapping class names to indices (optional)
    """
    def __init__(self, df, img_dirs, transforms=None, class_to_idx=None):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.img_dirs = img_dirs if isinstance(img_dirs, list) else [img_dirs]
        self.transforms = transforms

        if class_to_idx:
            self.class_to_idx = class_to_idx
        else:
            self.class_to_idx = CLASS_TO_IDX

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        img_id = str(row['ID'])
        label = self.class_to_idx[row['labels']]

        # Find image
        img_path = self._find_image(img_id)
        image = Image.open(img_path).convert('RGB')
        if self.transforms:
            image = self.transforms(image)

        return image, label

    def _find_image(self, img_id):
        """Find image path in available directories."""
        for img_dir in self.img_dirs:
            path = os.path.join(img_dir, img_id)
            if os.path.exists(path):
                return path
        raise FileNotFoundError(f"Image not found for ID {img_id}")


def boost_rare_classes(df, classes_to_boost, boost_factor=5):
    """
    Duplicate samples of rare classes to balance the dataset.

    Args:
        df: DataFrame with 'ID' and 'labels' columns
        classes_to_boost: List of class names to oversample (e.g., ['PC', 'PLY'])
        boost_factor: How many times to duplicate (5 = 5x more samples)

    Returns:
        DataFrame with boosted samples
    """
    print(f'Boosting rare classes: {classes_to_boost} by {boost_factor}x')

    df_to_boost = df[df['labels'].isin(classes_to_boost)]
    df_normal = df[~df['labels'].isin(classes_to_boost)]

    for cls in classes_to_boost:
        count_before = len(df[df['labels'] == cls])
        print(f'{cls}: {count_before} -> {count_before * boost_factor} samples')

    df_boosted = pd.concat([df_to_boost] * boost_factor, ignore_index=True)
    df_result = pd.concat([df_normal, df_boosted], ignore_index=True)

    print(f'Total: {len(df)} -> {len(df_result)} samples')

    return df_result


def load_dataframes():
    """Load all dataframes."""
    df_phase1 = pd.read_csv(LABEL_PATH_PHASE_1).iloc[:, :2]
    df_phase1.columns = ["ID", "labels"]

    df_phase2_train = pd.read_csv(LABEL_PATH_PHASE_2_TRAIN)[["ID", "labels"]]
    df_phase2_val = pd.read_csv(LABEL_PATH_PHASE_2_EVAL)[["ID", "labels"]]
    df_phase2_test = pd.read_csv(LABEL_PATH_PHASE_2_TEST)[["ID"]]

    return {
        "phase1": df_phase1,
        "phase2_train": df_phase2_train,
        "phase2_val": df_phase2_val,
        "phase2_test": df_phase2_test,
    }


def prepare_stage_data(dfs, config):
    """Prepare dataset for training."""
    train_transform = create_transforms(config, split="train")
    val_transform = create_transforms(config, split="val")

    # Get image directories
    base_dirs = [DIR_PHASE_1, DIR_PHASE_2_TRAIN, DIR_PHASE_2_EVAL]

    df = pd.concat([dfs["phase1"], dfs["phase2_train"], dfs["phase2_val"]], ignore_index=True)

    class_to_idx = CLASS_TO_IDX

    # Boost rare classes if enabled
    if config['data']['boost_rare_classes']['enable']:
        df = boost_rare_classes(
            df, 
            config['data']['boost_rare_classes']['rare_classes'], 
            config['data']['boost_rare_classes']['boost_factor']
        )

        print(f"Class distribution after boosting")
        for cls in CLASS_NAMES:
            count = len(df[df["labels"] == cls])
            marker = "🔺 BOOSTED" if cls in config['data']['boost_rare_classes']['rare_classes'] else ""
            print(f"   {cls}: {count} samples {marker}")

    # Split data
    df_train, df_val = train_test_split(
        df,
        test_size=config['data']["val_split"],
        random_state=config['data']["seed"],
        stratify=df["labels"]
    )

    # Compute samples per class
    train_labels = df['labels'].map(CLASS_TO_IDX).values
    samples_per_class = np.array([(train_labels == i).sum() for i in range(NUM_CLASSES)])

    # Create datasets
    train_dataset = WBCDataset(df_train, base_dirs, train_transform, class_to_idx)
    val_dataset = WBCDataset(df_val, base_dirs, val_transform, class_to_idx)

    print(f"Train={len(train_dataset)}, Val={len(val_dataset)}, Classes={len(class_to_idx)}")
    return train_dataset, val_dataset, samples_per_class, df_train, df_val


# Load data (using config1 for data preparation - same for both models)
print(f"Loading data...")
dfs = load_dataframes()

train_dataset, val_dataset, samples_per_class, df_train, df_val = prepare_stage_data(dfs, config1)
print("Data loading completed!")

# 4. Init Model, Optimizer, Loss function

## 4.1 Model

In [ ]:
def get_in_features(model):
    """Get the number of input features for the classifier head."""
    # ResNet
    if hasattr(model, 'fc'):
        return model.fc.in_features
    # ViT
    if hasattr(model, 'heads') and hasattr(model.heads, 'head'):
        return model.heads.head.in_features
    # ConvNeXt
    if hasattr(model, 'classifier'):
        return model.classifier[-1].in_features
    # Swin
    if hasattr(model, 'head'):
        return model.head.in_features

    raise ValueError(f"Cannot infer in_features")


def create_head(in_features, config):
    """Create classifier head based on configuration."""
    head_type = config['model']['head']['type']
    num_classes = config['model']['num_classes']

    if head_type == 'default':
        head = nn.Linear(in_features, num_classes)
    elif head_type == 'custom':
        hidden_dim = config['model']['head']['hidden_dim']
        dropout_rate = config['model']['head']['dropout']
        head = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, num_classes),
        )
    else:
        raise ValueError(f"Unknown head_type: {head_type}")

    # Initialize parameters
    for module in head.modules():
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight)
            nn.init.constant_(module.bias, 0)

    return head


def replace_head(model, new_head):
    """Replace the classifier head of the model."""
    base_model = model.module if hasattr(model, 'module') else model
    
    # ResNet
    if hasattr(base_model, 'fc') and isinstance(base_model.fc, nn.Module):
        base_model.fc = new_head
        return
        
    # ViT
    if hasattr(base_model, 'heads') and hasattr(base_model.heads, 'head'):
        base_model.heads.head = new_head
        return
        
    # Swin
    if hasattr(base_model, 'head'):
        base_model.head = new_head
        return
        
    # ConvNeXt
    if hasattr(base_model, 'classifier'):
        if isinstance(base_model.classifier, nn.Sequential):
            base_model.classifier[-1] = new_head
            return
            
    raise ValueError(f"Unsupported model for head replacement")


def create_model(config):
    """Create model based on configuration."""
    model_name = config['model']['name']
    pretrained = config['model']['pretrained']

    if model_name == 'resnet50':
        from torchvision.models import resnet50, ResNet50_Weights
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)

    elif model_name == 'resnet101':
        from torchvision.models import resnet101, ResNet101_Weights
        weights = ResNet101_Weights.DEFAULT if pretrained else None
        model = resnet101(weights=weights)

    elif model_name == 'resnet152':
        from torchvision.models import resnet152, ResNet152_Weights
        weights = ResNet152_Weights.DEFAULT if pretrained else None
        model = resnet152(weights=weights)

    elif model_name == 'vit_b_16':
        from torchvision.models import vit_b_16, ViT_B_16_Weights
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        model = vit_b_16(weights=weights)

    elif model_name == 'swin_tiny':
        from torchvision.models import swin_t, Swin_T_Weights
        weights = Swin_T_Weights.DEFAULT if pretrained else None
        model = swin_t(weights=weights)

    elif model_name == 'convnext_tiny':
        from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
        weights = ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        model = convnext_tiny(weights=weights)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    in_features = get_in_features(model)
    model._in_features = in_features
    head = create_head(in_features, config)
    replace_head(model, head)

    return model


print("✅ Model creation functions defined!")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]


Using DataParallel with 2 GPUs
Model: resnet50
Total params: 24,564,813
Trainable params: 24,564,813 (100.0%)


## 4.2 Loss function, optimizer, scheduler

In [ ]:
class CombinedLoss(nn.Module):
    """Combined Cross-Entropy and Focal Loss."""
    
    def __init__(self, weight=None, gamma=2.0, mix_ratio=0.5):
        """
        Args:
            weight (Tensor): Class weights (computed from effective_num).
            gamma (float): Focal Loss gamma parameter.
            mix_ratio (float): Mixing ratio (0.0 -> 1.0). 
                               0.5 means 50% CE + 50% Focal.
        """
        super(CombinedLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight
        self.mix_ratio = mix_ratio
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        
        # Compute Focal Loss component
        pt = torch.exp(-ce_loss) 
        focal_term = (1 - pt) ** self.gamma
        focal_loss = focal_term * ce_loss
        
        # Combine: (1 - ratio) * CE + (ratio) * Focal
        combined = (1 - self.mix_ratio) * ce_loss + self.mix_ratio * focal_loss
        
        return combined.mean()


def create_criterion(config, stage, samples_per_class, device):
    """Create loss criterion based on configuration."""
    loss_type = stage['loss']['name']

    if loss_type == 'cross_entropy':
        loss_function = nn.CrossEntropyLoss()

    elif loss_type == 'weighted_cross_entropy':
        beta = stage['loss'].get('beta', 0.9999)
        
        samples_per_class = np.array(samples_per_class)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / np.clip(effective_num, a_min=1e-8, a_max=None)
        weights = weights / weights.mean()
        weights = torch.tensor(weights, dtype=torch.float).to(device)

        loss_function = nn.CrossEntropyLoss(weight=weights)

    elif loss_type == 'combined_loss':
        beta = stage['loss'].get('beta', 0.9999)
        samples_per_class = np.array(samples_per_class)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / np.clip(effective_num, a_min=1e-8, a_max=None)
        weights = weights / weights.mean()
        weights = torch.tensor(weights, dtype=torch.float).to(device)

        gamma = stage['loss'].get('gamma', 2.0)
        mix_ratio = stage['loss'].get('mix_ratio', 0.5)  # Default 50-50

        loss_function = CombinedLoss(weight=weights, gamma=gamma, mix_ratio=mix_ratio)

    else:
        raise ValueError(f"Unknown loss function name: {loss_type}")

    print(f"Using {loss_function.__class__.__name__}")

    return loss_function


def create_optimizer(model, stage):
    """Create optimizer based on configuration."""
    optimizer_name = stage['optimizer']['name']
    lr = stage['optimizer']['lr']
    weight_decay = stage['optimizer']['weight_decay']

    if optimizer_name == 'SGD':
        optimizer = torch.optim.SGD(model.parameters(), momentum=0.9,
                                    lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer name: {optimizer_name}")

    print(f"Using {optimizer.__class__.__name__} as optimizer")

    return optimizer


def create_scheduler(optimizer, stage):
    """Create learning rate scheduler."""
    T_max = stage['epochs']
    scheduler = CosineAnnealingLR(optimizer, T_max=T_max, eta_min=0)

    return scheduler


def create_sampler(dataset, stage, df):
    """Create data sampler based on configuration."""
    sampler_type = stage['sampler']['name']
    replacement = stage['sampler']['replacement']

    labels = df['labels'].map(CLASS_TO_IDX).values
    num_samples = len(labels)

    if sampler_type == 'random_sampler':
        sampler = torch.utils.data.RandomSampler(dataset, replacement=replacement)

    elif sampler_type == 'class_balanced_sampler':
        class_count = np.bincount(labels)
        class_weights = 1.0 / class_count

        sample_weights = class_weights[labels]
        sample_weights = torch.DoubleTensor(sample_weights)

        sampler = torch.utils.data.WeightedRandomSampler(
            weights=sample_weights,
            num_samples=num_samples,
            replacement=replacement
        )

    elif sampler_type == 'square_root_sampler':
        class_counts = np.bincount(labels)
        class_weights = 1.0 / np.sqrt(class_counts)

        sample_weights = class_weights[labels]
        sample_weights = torch.DoubleTensor(sample_weights)

        sampler = torch.utils.data.WeightedRandomSampler(
            weights=sample_weights,
            num_samples=num_samples,
            replacement=replacement
        )

    else:
        raise ValueError(f"Unknown sampler name: {sampler_type}")

    print(f"Using {sampler_type}")
    return sampler


print(f"✅ Loss, optimizer, scheduler functions defined!")

Define function, optimizer, scheduler completely!


## 4.3 Training utils

In [ ]:
def unwrap_model(model):
    """Unwrap model from DataParallel if necessary."""
    return model.module if hasattr(model, 'module') else model

    
def freeze_backbone(model, config):
    """Freeze backbone and reinitialize classifier head."""
    base_model = unwrap_model(model)
    device = next(base_model.parameters()).device
    
    # Freeze all parameters
    for param in base_model.parameters():
        param.requires_grad = False

    # Reinitialize classifier weights
    in_features = base_model._in_features
    new_head = create_head(in_features, config).to(device)
    replace_head(base_model, new_head)

    # Enable gradients only for head
    for param in new_head.parameters():
        param.requires_grad = True

    return model


def compute_metrics(y_true, y_pred, class_names):
    """
    Compute all metrics using sklearn.

    Args:
        y_true: Ground truth labels (as indices)
        y_pred: Predicted labels (as indices)
        class_names: List of class names for this stage
    """
    labels = list(range(len(class_names)))

    metrics = {
        'macro_f1': f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0) * 100,
        'macro_precision': precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0) * 100,
        'macro_recall': recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0) * 100,
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred) * 100,
        'accuracy': (np.array(y_true) == np.array(y_pred)).mean() * 100,
    }

    # Per-class F1
    per_class_f1 = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0) * 100
    for i, name in enumerate(class_names):
        metrics[f"f1_{name}"] = per_class_f1[i] if i < len(per_class_f1) else 0.0

    return metrics


def print_per_class_f1(y_true, y_pred, class_names, stage_name, sort_by_f1=True):
    """
    Print F1 score for each class after training a stage.
    
    Args:
        y_true: Ground truth labels (as indices)
        y_pred: Predicted labels (as indices)
        class_names: List of class names
        stage_name: Name of the current stage
        sort_by_f1: If True, sort by F1 descending; else keep original order
    """
    labels = list(range(len(class_names)))
    
    # Compute F1 score for each class
    per_class_f1 = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0) * 100
    per_class_precision = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0) * 100
    per_class_recall = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0) * 100
    
    # Compute support (actual sample count for each class in y_true)
    support_count = Counter(y_true)
    
    # Create list with info for each class
    class_stats = []
    for i, name in enumerate(class_names):
        class_stats.append({
            'name': name,
            'f1': per_class_f1[i] if i < len(per_class_f1) else 0.0,
            'precision': per_class_precision[i] if i < len(per_class_precision) else 0.0,
            'recall': per_class_recall[i] if i < len(per_class_recall) else 0.0,
            'support': support_count.get(i, 0)
        })
    
    # Sort by F1 descending
    if sort_by_f1:
        class_stats = sorted(class_stats, key=lambda x: x['f1'], reverse=True)
    
    # Print results
    print(f"\n{'='*80}")
    print(f"📊 PER-CLASS F1 SCORES - STAGE {stage_name.upper()}")
    print(f"{'='*80}")
    print(f"{'Rank':<6}{'Class':<30}{'F1':>10}{'Precision':>12}{'Recall':>10}{'Support':>10}")
    print(f"{'-'*80}")
    
    for rank, stats in enumerate(class_stats, 1):
        # Highlight classes with low F1 (< 50%)
        if stats['f1'] < 50:
            indicator = "⚠️"
        elif stats['f1'] >= 80:
            indicator = "✅"
        else:
            indicator = "  "
        
        print(f"{rank:<6}{stats['name']:<30}{stats['f1']:>8.2f}%{stats['precision']:>10.2f}%{stats['recall']:>8.2f}%{stats['support']:>10} {indicator}")
    
    print(f"{'-'*80}")
    
    # Compute and print summary statistics
    f1_values = [s['f1'] for s in class_stats]
    print(f"{'SUMMARY':<36}{'Mean':>10}{'Std':>12}{'Min':>10}{'Max':>10}")
    print(f"{'F1 Score':<36}{np.mean(f1_values):>8.2f}%{np.std(f1_values):>10.2f}%{np.min(f1_values):>8.2f}%{np.max(f1_values):>8.2f}%")
    
    # Count classes with F1 < threshold
    low_f1_classes = [s for s in class_stats if s['f1'] < 50]
    if low_f1_classes:
        print(f"\n⚠️  {len(low_f1_classes)} classes with F1 < 50%:")
        for s in low_f1_classes[:5]:  # Show top 5 worst classes
            print(f"   - {s['name']}: {s['f1']:.2f}% (support: {s['support']})")
        if len(low_f1_classes) > 5:
            print(f"   ... and {len(low_f1_classes) - 5} other classes")
    
    print(f"{'='*80}\n")
    
    return class_stats

def save_checkpoint(model, optimizer, scheduler, epoch, metrics, filename):
    """Save checkpoint."""
    state = {
        'epoch': epoch,
        'model': model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
        'optimizer': optimizer.state_dict() if optimizer else None,
        'scheduler': scheduler.state_dict() if scheduler else None,
        'metrics': metrics,
    }
    torch.save(state, filename)


def load_checkpoint_for_training(model, optimizer, scheduler, filename, device):
    """Load checkpoint for continuing training."""
    checkpoint = torch.load(filename, map_location=device, weights_only=False)

    if hasattr(model, 'module'):
        model.module.load_state_dict(checkpoint['model'])
    else:
        model.load_state_dict(checkpoint['model'])

    if optimizer and checkpoint.get('optimizer'):
        optimizer.load_state_dict(checkpoint['optimizer'])
    if scheduler and checkpoint.get('scheduler'):
        scheduler.load_state_dict(checkpoint['scheduler'])

    return checkpoint['epoch'], checkpoint['metrics']


def train_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        # Delete unnecessary tensors
        del images, labels, outputs, loss, preds

    avg_loss = total_loss / total
    acc = correct / total * 100

    return avg_loss, acc


def validate(model, loader, criterion, device, class_names):
    """Validate model."""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.inference_mode():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images, labels = images.to(device), labels.to(device)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            # Convert to list
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            # Delete tensors
            del images, labels, outputs, loss, preds

        avg_loss = total_loss / len(loader.dataset)
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)

        metrics = compute_metrics(all_labels, all_preds, class_names)
        metrics['loss'] = avg_loss

    return metrics, all_preds, all_labels


def train_stage(model, train_df, val_df, train_dataset, val_dataset,
                stage, class_names, config, device, g, samples_per_class, start_epoch=0):
    """
    Train one stage.
    """
    batch_size = config['data']['batch_size']
    num_workers = config['data']['num_workers']

    stage_name = stage['name']
    num_epochs = stage['epochs']
    lr = stage['optimizer']['lr']

    early_stopping = stage['early_stopping']['enable']
    patience = stage['early_stopping'].get('patience', None)

    print(f"\n{'='*70}")
    print(f"🚀 STAGE {stage_name.upper()} TRAINING")
    print(f"{'='*70}")
    print(f"   Epochs (stage): {num_epochs}")
    print(f"   Start global epoch: {start_epoch}")
    print(f"   LR: {lr}")
    print(f"   Classes: {len(class_names)}")
    print(f"   Early stopping: {early_stopping}, Patience: {patience}")

    scaler = torch.amp.GradScaler()

    # Freeze/unfreeze backbone
    if stage['freeze_backbone']:
        model = freeze_backbone(model, config)

    total_params = sum(
        param.numel() for param in model.parameters() if param.requires_grad
    )
    print(f"\nTotal trainable params: {total_params:,}")

    # Sampler & DataLoader
    train_sampler = create_sampler(train_dataset, stage, df=train_df)

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              sampler=train_sampler, num_workers=num_workers,
                              drop_last=True, worker_init_fn=seed_worker if num_workers > 0 else None,
                              generator=g if num_workers > 0 else None, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, worker_init_fn=seed_worker if num_workers > 0 else None,
                            generator=g if num_workers > 0 else None, pin_memory=True)

    # Loss, Optimizer & Scheduler
    criterion = create_criterion(config, stage, samples_per_class, device)
    optimizer = create_optimizer(model, stage)
    scheduler = create_scheduler(optimizer, stage)

    # Training state
    best_metric = 0
    best_epoch = start_epoch
    no_improve = 0

    # Variables to store last predictions for confusion matrix
    last_val_preds = None
    last_val_labels = None

    for global_epoch in tqdm(
        range(start_epoch, start_epoch + num_epochs),
        desc=f"Stage {stage_name}",
        ):

        stage_epoch = global_epoch - start_epoch
        epoch_start = time()

        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)

        # Validate
        val_metrics, val_preds, val_labels = validate(model, val_loader, criterion, device, class_names)

        # Store last predictions
        last_val_preds = val_preds
        last_val_labels = val_labels

        # Step scheduler
        scheduler.step()

        # Logging
        current_lr = optimizer.param_groups[0]['lr']
        epoch_time = time() - epoch_start

        # Print progress
        print(f"\nEpoch {stage_epoch+1}/{num_epochs} [{epoch_time:.1f}s]")
        print(f"   Train - Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
        print(f"   Val   - Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.2f}%")
        print(f"   🎯 Macro F1: {val_metrics['macro_f1']:.2f}% | Balanced Acc: {val_metrics['balanced_accuracy']:.2f}%")
        print(f"   LR: {current_lr:.2e}")

        # Best model check
        if val_metrics['macro_f1'] > best_metric:
            best_metric = val_metrics['macro_f1']
            best_epoch = global_epoch
            no_improve = 0

            save_checkpoint(model, optimizer, scheduler, global_epoch, val_metrics,
                          os.path.join(SAVE_DIR, f"best_stage_{stage_name}.pt"))
            print(f"   🏆 New best! Macro F1: {best_metric:.2f}%")
        else:
            no_improve += 1
            print(f"   Epochs without improvement: {no_improve}/{patience}")

        # Early stopping
        if early_stopping and no_improve >= patience:
            print(
                f"⚠️ Early stopping stage {stage_name} "
                f"at stage_epoch {stage_epoch+1}"
            )
            break

        # Clear memory after each epoch
        del val_metrics
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # Print per-class F1 at end of stage
    if last_val_preds is not None:
        print_per_class_f1(last_val_labels, last_val_preds, class_names, stage_name)
        
    print(f"\n✅ Stage {stage_name} completed")
    print(f"   Best Macro F1: {best_metric:.2f}% @ epoch {best_epoch+1}")

    # Final cleanup
    del optimizer, scheduler, last_val_preds, last_val_labels
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return global_epoch + 1, best_metric


print("✅ Training functions defined!")

✅ Training functions defined!


# 5. Training loop

In [ ]:
# Store trained models info
trained_models = {}
model_names_list = []

for model_name, config in CONFIGS:
    print(f"\n{'#'*80}")
    print(f"# TRAINING MODEL: {model_name.upper()}")
    print(f"{'#'*80}")
    
    # Create model
    model = create_model(config)
    
    if torch.cuda.device_count() > 1:
        print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    
    model = model.to(device)
    
    # Count parameters
    total_params = sum(module.numel() for module in model.parameters())
    trainable_params = sum(module.numel() for module in model.parameters() if module.requires_grad)
    print(f"Model: {config['model']['name']}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,} ({trainable_params*100/total_params:.1f}%)")
    
    # Train all stages
    global_epoch = 0
    best_metrics = {}
    
    for stage in config['training']['stages']:
        global_epoch, best_f1 = train_stage(
            model=model,
            train_df=df_train,
            val_df=df_val,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            stage=stage,
            class_names=CLASS_NAMES,
            config=config,
            device=device,
            g=g,
            samples_per_class=samples_per_class,
            start_epoch=global_epoch
        )
        best_metrics[stage["name"]] = best_f1
    
    # Save final model
    final_metric = list(best_metrics.values())[-1]
    checkpoint_path = os.path.join(SAVE_DIR, f'final_model_{model_name}.pt')
    save_checkpoint(model, None, None, -1, {'macro_f1': final_metric}, checkpoint_path)
    
    # Store model info
    trained_models[model_name] = {
        'checkpoint_path': checkpoint_path,
        'config': config,
        'best_metrics': best_metrics,
    }
    model_names_list.append(model_name)
    
    # Summary
    print(f"\n{'='*70}")
    print(f"🎉 TRAINING COMPLETED FOR {model_name.upper()}!")
    print(f"{'='*70}")
    for stage_name, metric in best_metrics.items():
        print(f"   Stage {stage_name} Best Macro F1: {metric:.2f}%")
    print(f"   Saved to: {checkpoint_path}")
    
    # Cleanup
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# Final summary
print(f"\n{'#'*80}")
print(f"# ALL MODELS TRAINING COMPLETED!")
print(f"{'#'*80}")
for name in model_names_list:
    info = trained_models[name]
    print(f"\n{name}:")
    for stage_name, metric in info['best_metrics'].items():
        print(f"   Stage {stage_name}: {metric:.2f}%")


🚀 STAGE BACKBONE TRAINING
   Epochs (stage): 100
   Start global epoch: 0
   LR: 0.0001
   Classes: 13
   Early stopping: True

Total trainable params: 24,564,813
Use random_sampler
Use CrossEntropyLoss
Use AdamW as optimizer


Stage backbone:   0%|          | 0/100 [00:00<?, ?it/s]

Training:   0%|          | 0/120 [00:00<?, ?it/s]

Validating:   0%|          | 0/31 [00:00<?, ?it/s]


Epoch 1/100 [222.9s]
   Train - Loss: 0.7313 | Acc: 79.07%
   Val   - Loss: 0.3453 | Acc: 89.89%
   🎯 Macro F1: 50.75% | Balanced Acc: 48.20%
   LR: 1.00e-04
   🏆 New best! Macro F1: 50.75%


...

Validating:   0%|          | 0/31 [00:00<?, ?it/s]


Epoch 27/50 [114.8s]
   Train - Loss: 0.0010 | Acc: 99.24%
   Val   - Loss: 0.0549 | Acc: 91.41%
   🎯 Macro F1: 66.76% | Balanced Acc: 72.50%
   LR: 4.37e-04
Epoch no improved: 9/10


Training:   0%|          | 0/120 [00:00<?, ?it/s]

Validating:   0%|          | 0/31 [00:00<?, ?it/s]


Epoch 28/50 [112.0s]
   Train - Loss: 0.0014 | Acc: 99.13%
   Val   - Loss: 0.0590 | Acc: 91.59%
   🎯 Macro F1: 67.77% | Balanced Acc: 73.73%
   LR: 4.06e-04
Epoch no improved: 10/10
⚠️ Early stopping stage classifier at stage_epoch 28

📊 PER-CLASS F1 SCORES - STAGE CLASSIFIER
Rank  Class                                 F1   Precision    Recall   Support
--------------------------------------------------------------------------------
1     SNE                              97.41%     98.24%   96.60%      3471 ✅
2     EO                               94.95%     96.00%   93.91%       230 ✅
3     LY                               94.83%     96.23%   93.47%      2160 ✅
4     BL                               88.10%     88.51%   87.69%       536 ✅
5     BA                               87.83%     84.87%   90.99%       111 ✅
6     MO                               87.80%     87.62%   87.98%       732 ✅
7     PC                               60.61%     66.67%   55.56%        18   
8     MY      